# MIDI Feature Extraction

## Purpose

This notebook creates and enriches the composer-classification feature datasets for Bach, Beethoven, Chopin, and Mozart. Each CSV row represents one MIDI file.

## Repository Paths

This notebook lives in the repository's notebooks folder. Run it with notebooks as the current working directory so the relative paths resolve correctly:

```powershell
Set-Location notebooks
jupyter notebook
```

| Resource | Path from this notebook |
| --- | --- |
| Source MIDI files | `../data/midiclassics` |
| Training features | `../data/features/train_features.csv` |
| Development features | `../data/features/dev_features.csv` |
| Test features | `../data/features/test_features.csv` |

## Feature Dataset

The current CSVs contain 63 columns: `composer`, `filename`, `relative_path`, plus 60 numeric descriptors.

- **Identifier fields:** composer, filename, and relative_path. These support labeling and traceability; do not use them as model inputs.
- **Musical descriptors:** pitch, timing, dynamics, chords, note density, onset intervals, and polyphony.
- **Instrumentation descriptors:** MIDI-track counts, drum presence, unique program counts, and General MIDI family track counts.

## Idempotent Workflow

1. **Base extraction:** runs only if one or more split CSVs is absent. It parses MIDI files and runs the slower music21 chord analysis.
2. **Feature enrichment:** checks every derived and MIDI-level feature column. A complete column is skipped; only missing or incomplete columns are calculated and written.

This preserves the current train/dev/test assignments and avoids recreating the dataset when adding a future feature. Some MIDI files have malformed tempo, key, or time-signature events, so those metadata may be imperfect.


In [1]:
import os
import pandas as pd
import numpy as np

import music21
import pretty_midi
from music21 import converter, chord

In [2]:
BASE_DIR = "../data/midiclassics"
OUTPUT_DIR = "../data/features"
SPLIT_RATIOS = {"train": 0.70, "dev": 0.15, "test": 0.15}

COMPOSERS = [
    "Bach",
    "Beethoven",
    "Chopin",
    "Mozart"
]

def extract_features(filepath, composer):
    try:
        midi = pretty_midi.PrettyMIDI(filepath)

        notes = []
        durations = []
        velocities = []

        for instrument in midi.instruments:
            for note in instrument.notes:
                notes.append(note.pitch)
                durations.append(note.end - note.start)
                velocities.append(note.velocity)

        if len(notes) == 0:
            return None

        tempo = midi.estimate_tempo()

        pitch_hist = midi.get_pitch_class_histogram()

        note_density = len(notes) / midi.get_end_time()

        avg_pitch = np.mean(notes)
        pitch_range = np.max(notes) - np.min(notes)
        avg_duration = np.mean(durations)
        avg_velocity = np.mean(velocities)

        ####################################################
        # Count chords using music21
        ####################################################

        score = converter.parse(filepath)

        chords = score.chordify()

        chord_count = 0

        for c in chords.recurse().getElementsByClass(chord.Chord):
            chord_count += 1

        ####################################################

        features = {
            "composer": composer,
            "filename": os.path.basename(filepath),
            "tempo": tempo,
            "num_notes": len(notes),
            "num_chords": chord_count,
            "avg_pitch": avg_pitch,
            "pitch_range": pitch_range,
            "avg_duration": avg_duration,
            "avg_velocity": avg_velocity,
            "note_density": note_density,
        }

        for i in range(12):
            features[f"pitch_class_{i}"] = pitch_hist[i]

        return features

    except Exception as e:
        print(f"Error reading {filepath}")
        print(e)
        return None


def split_files(files, rng):
    files = list(files)
    rng.shuffle(files)

    train_end = int(len(files) * SPLIT_RATIOS["train"])
    dev_end = train_end + int(len(files) * SPLIT_RATIOS["dev"])

    return {
        "train": files[:train_end],
        "dev": files[train_end:dev_end],
        "test": files[dev_end:],
    }


def process_dataset():
    rows_by_split = {split: [] for split in SPLIT_RATIOS}
    rng = np.random.default_rng(42)

    for composer in COMPOSERS:
        composer_folder = os.path.join(BASE_DIR, composer)
        midi_files = [
            os.path.join(root, file)
            for root, _, files in os.walk(composer_folder)
            for file in files
            if file.lower().endswith((".mid", ".midi"))
        ]

        if not midi_files:
            print(f"No MIDI files found for {composer}; skipping.")
            continue

        print(f"Processing {composer}: {len(midi_files)} files")
        for split, files in split_files(midi_files, rng).items():
            for filepath in files:
                features = extract_features(filepath, composer)
                if features is not None:
                    rows_by_split[split].append(features)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for split, rows in rows_by_split.items():
        output = os.path.join(OUTPUT_DIR, f"{split}_features.csv")
        pd.DataFrame(rows).to_csv(output, index=False)
        print(f"Saved {len(rows)} rows to {output}")


expected_splits = [os.path.join(OUTPUT_DIR, f"{split}_features.csv") for split in SPLIT_RATIOS]
if all(os.path.exists(path) for path in expected_splits):
    print("Existing feature CSVs found; skipping base extraction and preserving current splits.")
else:
    process_dataset()

Existing feature CSVs found; skipping base extraction and preserving current splits.


In [3]:
# Idempotently enrich existing split CSVs without recreating or re-splitting them.
import mido

GM_FAMILIES = [
    "piano", "chromatic_percussion", "organ", "guitar",
    "bass", "strings", "ensemble", "brass",
    "reed", "pipe", "synth_lead", "synth_pad",
    "synth_effects", "ethnic", "percussive", "sound_effects",
]

def value_is_missing(value):
    if pd.isna(value):
        return True
    return isinstance(value, str) and not value.strip()

def column_is_complete(df, column):
    return column in df.columns and not df[column].map(value_is_missing).any()

def set_missing_values(df, column, values):
    if column not in df.columns:
        df[column] = values
        return
    missing = df[column].map(value_is_missing)
    if missing.any():
        df.loc[missing, column] = values.loc[missing]

def add_dataframe_features(df):
    """Add derived features only when a column is missing or incomplete."""
    pitch_columns = [f"pitch_class_{i}" for i in range(12)]
    pitch_values = df[pitch_columns].to_numpy(dtype=float)
    pitch_probabilities = pitch_values / (pitch_values.sum(axis=1, keepdims=True) + 1e-10)
    chromatic_indices = [1, 3, 6, 8, 10]

    derived = {
        "pitch_entropy": -np.sum(pitch_probabilities * np.log(pitch_probabilities + 1e-10), axis=1),
        "pitch_class_variance": np.var(pitch_values, axis=1),
        "range_normalized": df["pitch_range"] / (df["avg_pitch"] + 1e-6),
        "notes_per_chord": df["num_notes"] / (df["num_chords"] + 1),
        "chord_density": df["num_chords"] / (df["num_notes"] + 1),
        "velocity_variation": df["avg_velocity"] / (df["tempo"] + 1),
        "tempo_note_ratio": df["tempo"] / (df["num_notes"] + 1),
        "chromatic_ratio": pitch_values[:, chromatic_indices].sum(axis=1) / (pitch_values.sum(axis=1) + 1e-6),
    }
    for column, values in derived.items():
        if not column_is_complete(df, column):
            set_missing_values(df, column, pd.Series(values, index=df.index))

def build_midi_index():
    """Map each unique (composer, filename) pair to its source MIDI path."""
    index = {}
    for composer in COMPOSERS:
        composer_folder = os.path.join(BASE_DIR, composer)
        for root, _, files in os.walk(composer_folder):
            for filename in files:
                if not filename.lower().endswith((".mid", ".midi")):
                    continue
                key = (composer, filename.casefold())
                if key in index:
                    raise ValueError(f"Duplicate source filename for {composer}: {filename}")
                index[key] = os.path.join(root, filename)
    return index

def calculate_midi_features(filepath):
    """Extract fast MIDI-level descriptors; this deliberately skips music21 chordification."""
    midi = pretty_midi.PrettyMIDI(filepath)
    note_tracks = [instrument for instrument in midi.instruments if instrument.notes]
    notes = [note for instrument in note_tracks for note in instrument.notes]
    if not notes:
        raise ValueError("No notes found")

    pitches = np.array([note.pitch for note in notes], dtype=float)
    velocities = np.array([note.velocity for note in notes], dtype=float)
    durations = np.array([note.end - note.start for note in notes], dtype=float)
    onsets = np.sort(np.array([note.start for note in notes], dtype=float))
    onset_intervals = np.diff(onsets)
    total_duration = midi.get_end_time()

    events = sorted(
        [(note.start, 1) for note in notes] + [(note.end, -1) for note in notes],
        key=lambda event: event[0],
    )
    active_notes = 0
    max_polyphony = 0
    weighted_polyphony = 0.0
    previous_time = events[0][0]
    event_index = 0
    while event_index < len(events):
        event_time = events[event_index][0]
        weighted_polyphony += active_notes * (event_time - previous_time)
        while event_index < len(events) and events[event_index][0] == event_time:
            active_notes += events[event_index][1]
            event_index += 1
        max_polyphony = max(max_polyphony, active_notes)
        previous_time = event_time

    features = {
        "relative_path": os.path.relpath(filepath, BASE_DIR),
        "total_duration": total_duration,
        "num_midi_tracks": len(mido.MidiFile(filepath).tracks),
        "num_note_tracks": len(note_tracks),
        "num_unique_programs": len({instrument.program for instrument in note_tracks if not instrument.is_drum}),
        "drum_track_count": sum(instrument.is_drum for instrument in note_tracks),
        "has_drums": int(any(instrument.is_drum for instrument in note_tracks)),
        "pitch_std": np.std(pitches),
        "pitch_median": np.median(pitches),
        "velocity_std": np.std(velocities),
        "velocity_range": np.ptp(velocities),
        "duration_std": np.std(durations),
        "duration_median": np.median(durations),
        "onset_interval_mean": np.mean(onset_intervals) if len(onset_intervals) else 0.0,
        "onset_interval_std": np.std(onset_intervals) if len(onset_intervals) else 0.0,
        "max_polyphony": max_polyphony,
        "avg_polyphony": weighted_polyphony / total_duration if total_duration else 0.0,
    }
    for family in GM_FAMILIES:
        features["gm_" + family + "_track_count"] = 0
    for instrument in note_tracks:
        if not instrument.is_drum:
            features["gm_" + GM_FAMILIES[instrument.program // 8] + "_track_count"] += 1
    return features

def enrich_existing_csvs():
    split_paths = [os.path.join(OUTPUT_DIR, f"{split}_features.csv") for split in ["train", "dev", "test"]]
    dataframes = {csv_path: pd.read_csv(csv_path) for csv_path in split_paths}

    for df in dataframes.values():
        add_dataframe_features(df)

    midi_columns = [
        "relative_path", "total_duration", "num_midi_tracks", "num_note_tracks",
        "num_unique_programs", "drum_track_count", "has_drums", "pitch_std",
        "pitch_median", "velocity_std", "velocity_range", "duration_std",
        "duration_median", "onset_interval_mean", "onset_interval_std",
        "max_polyphony", "avg_polyphony",
        *["gm_" + family + "_track_count" for family in GM_FAMILIES],
    ]
    needed_columns = [
        column for column in midi_columns
        if not all(column_is_complete(df, column) for df in dataframes.values())
    ]

    if needed_columns:
        print(f"Calculating missing or incomplete MIDI columns: {needed_columns}")
        midi_index = build_midi_index()
        for df in dataframes.values():
            for row_index, row in df[["composer", "filename"]].iterrows():
                missing_columns = [
                    column for column in needed_columns
                    if not column_is_complete(df.iloc[[row_index]], column)
                ]
                if not missing_columns:
                    continue
                filepath = midi_index.get((row["composer"], row["filename"].casefold()))
                if filepath is None:
                    raise FileNotFoundError(f"No source MIDI found for {row['composer']}: {row['filename']}")
                features = calculate_midi_features(filepath)
                for column in missing_columns:
                    df.at[row_index, column] = features[column]
    else:
        print("All MIDI enrichment columns are already complete; skipping MIDI parsing.")

    for csv_path, df in dataframes.items():
        df.to_csv(csv_path, index=False)
        print(f"Saved {csv_path}: {df.shape[1]} columns")

enrich_existing_csvs()


All MIDI enrichment columns are already complete; skipping MIDI parsing.
Saved ../data/features\train_features.csv: 63 columns
Saved ../data/features\dev_features.csv: 63 columns
Saved ../data/features\test_features.csv: 63 columns
